# Brewery clustering — every run of DBSCAN, HDBSCAN and OPTICS

The first clustering notebook (`brewery_clustering_revised.ipynb`) swept 264 DBSCAN and 336 HDBSCAN settings per
dataset but kept the labels of only five hand-picked candidates. This notebook keeps **every run**, so any setting
can be inspected on the map, compared with its neighbours and judged on geography rather than on a single number.

**What it does**

1. Builds the coordinate corrections (8 swapped coordinates put back, 1 wrong coordinate sign reversed, 1 misplaced
   duplicate fixed, 18 far-off records flagged) and writes them to `data/coordinate_corrections.csv`.
2. Runs the full DBSCAN grid (24 `eps` × 11 `min_samples`) and HDBSCAN grid (28 `min_cluster_size` × 12
   `min_samples`, including the default) on both datasets, plus OPTICS (11 `min_samples` × 3 `xi`).
3. Keeps what the HDBSCAN talk recommends looking at: the condensed tree, cluster persistence, GLOSH outlier
   scores, the core distance as a density estimate, and DBSCAN as a flat cut of the HDBSCAN tree.
4. Describes every run geographically: compactness, density, how many states and countries its clusters span,
   and whether a cluster joins two distinct brewery hubs.
5. Measures stability: does a run's grouping survive a one-step change of its parameters?
6. Stores everything in new `allruns_*` tables of `data/clustering_results.db`. The tables written by the first
   notebook are left untouched.

**Runtime** about 15 minutes on an 8-core CPU. The GPU is not used: the data has 9,488 points, every step is
already fast on the CPU, and the GPU libraries (cuML) have no Haversine distance, so they would change the
results instead of only speeding them up.

**Run** from the project root: `python -m jupyter nbconvert --to notebook --execute --inplace brewery_clustering_all_runs.ipynb`

## 1. Setup

In [1]:
# Libraries, project modules and the parameter grids.
import json
import sqlite3
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.cluster import HDBSCAN as SklearnHDBSCAN
from sklearn.metrics import adjusted_rand_score

import cluster_analysis as ca
import origins_core as oc
import quality_core as qc

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("data")
RESULTS_DB = DATA / "clustering_results.db"
DATASETS = {"all_records": ("analysis_all_records.db", "analysis_all_records"),
            "unique_records": ("analysis_unique_records.db", "analysis_unique_records")}

# Settings for the geographic reading of clusters. A "hub" is a city with at least HUB_MIN_RECORDS breweries; a
# cluster "merges distinct hubs" when at least two hubs each put HUB_MIN_INSIDE records in it and their centers
# are more than HUB_MIN_SEP_KM apart.
HUB_MIN_RECORDS, HUB_MIN_INSIDE, HUB_MIN_SEP_KM = 10, 3, 100
N_JOBS = 8

print("DBSCAN eps (km):", ca.EPS_KM)
print("min_samples:", ca.MIN_SAMPLES)
print("HDBSCAN min_cluster_size:", ca.MIN_CLUSTER_SIZE)
print("OPTICS xi:", ca.OPTICS_XI, "| OPTICS max_eps (km):", ca.OPTICS_MAX_EPS_KM)

DBSCAN eps (km): [25, 30, 35, 40, 45, 50, 55, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 280, 300, 320, 400, 480, 500, 520]
min_samples: [4, 5, 6, 8, 10, 12, 15, 20, 25, 30, 32]
HDBSCAN min_cluster_size: [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240, 280, 300, 320, 400, 480, 500, 520]
OPTICS xi: [0.01, 0.05, 0.1] | OPTICS max_eps (km): 520


## 2. Coordinate corrections

Every geocoded record is compared with the GeoNames center of the city it names (`data/reference_places.db`).

- More than 500 km from its city but within 25 km once latitude and longitude are exchanged → **swapped back**.
- More than 50 km from its city and an identical twin record (same values except id and coordinates) sits in the
  city → **copied from identical record**.
- More than 50 km from its city with no safe correction → **flagged only** (kept where it is, flagged).

The check runs on the raw snapshot, so it also covers records outside the analysis datasets. The databases are not
modified; the log is applied whenever data is loaded.

In [2]:
# Build the corrections log from the raw snapshot and save it next to the databases.
ref = oc.load_reference()
with sqlite3.connect(DATA / "raw_breweries.db") as con:
    raw = qc.harmonize_states(pd.read_sql_query("SELECT * FROM raw_breweries", con))
corrections = oc.build_corrections(raw, ref)
corrections.to_csv(qc.CORRECTIONS_FILE, index=False)
print(corrections["action"].value_counts().to_string())
corrections[["action", "name", "city", "country", "original_latitude", "original_longitude",
             "corrected_latitude", "corrected_longitude", "km_to_city_center_original",
             "km_to_city_center_corrected"]].round(3)

action
flagged only                    18
swapped back                     8
sign flipped back                1
copied from identical record     1


,action,name,city,country,original_latitude,original_longitude,corrected_latitude,corrected_longitude,km_to_city_center_original,km_to_city_center_corrected
0,swapped back,Labyrinth Brewing Company,Manchester,United States,-72.565,41.777,41.777,-72.565,15198.372,3.602
1,swapped back,Goat Island Brewing,Cullman,United States,-86.821,34.141,34.141,-86.821,13984.554,4.308
2,swapped back,Vadia,Oliveira de Azemeis,Portugal,-8.434,40.841,40.841,-8.434,7441.925,3.473
3,swapped back,Xarlie,Colmeias,Portugal,-8.722,39.829,39.829,-8.722,7345.875,3.228
4,swapped back,Brauerei Forst,Lagundo,Italy,11.117,46.678,46.678,11.117,5173.845,0.748
5,swapped back,Batzen,Bolzano,Italy,11.357,46.502,46.502,11.357,5115.069,1.831
6,swapped back,Floribunda,Salorno,Italy,11.216,46.243,46.243,11.216,5103.089,0.431
7,swapped back,Gassl Bräu,Chiusa,Italy,11.565,46.641,46.641,11.565,5100.892,0.103
8,sign flipped back,Mescan Brewery,Kilsallagh,Ireland,53.747,9.730,53.747,-9.730,NaN,NaN
9,flagged only,Armstrong Brewing Co,South San Francisco,United States,50.447,-119.193,50.447,-119.193,1445.105,1445.105


## 3. Load the corrected datasets

In [3]:
# Both analysis datasets with corrections applied and harmonized state labels. The record order fixed here is the
# order of every stored label array.
frames, X = {}, {}
for name, (db, table) in DATASETS.items():
    with sqlite3.connect(DATA / db) as con:
        df = pd.read_sql_query(f'SELECT * FROM "{table}"', con)
    df = qc.harmonize_states(qc.apply_corrections(df, corrections)).dropna(subset=["latitude", "longitude"])
    frames[name] = df.reset_index(drop=True)
    X[name] = ca.radians(frames[name])
    print(f"{name}: {len(df):,} records, {df['coordinate_status'].ne('as recorded').sum()} with a correction or flag")

all_records: 9,488 records, 28 with a correction or flag
unique_records: 9,487 records, 28 with a correction or flag


## 4. Method notes

**Distance.** Latitude and longitude are angles, so every algorithm uses the Haversine metric on radians and every
radius is given in kilometres and divided by the Earth's mean radius (6,371.0088 km). There is no feature scaling:
scaling would turn "45 km" into a number with no geographic meaning.

**One `eps` for the whole map.** A neighbourhood radius tuned per state would redefine "dense" from state to
state. It is like calling people tall above 170 cm in one city and above 190 cm in another: both cities could then
report "50% tall", but the two numbers would no longer measure the same thing. Every run here uses one radius for
every brewery, so clusters in different states are comparable.

**`min_samples`.** In scikit-learn a point counts itself among its neighbours; in the `hdbscan` library it does
not. This project always quotes scikit-learn's value and passes `min_samples − 1` to `hdbscan`, which builds the
*exact* minimum spanning tree (`approx_min_span_tree=False`; the library's faster approximate tree loses the exact
correspondence with DBSCAN checked in section 7). The HDBSCAN "default"
column means `min_samples = min_cluster_size`, as in scikit-learn.

**`min_cluster_size` (HDBSCAN).** The minimum number of breweries needed to form a cluster. Groups that shrink
below it while the density threshold rises are treated as points falling out of their parent cluster, not as new
clusters.

**OPTICS** (Ordering Points To Identify the Clustering Structure) addresses DBSCAN's fixed `eps`: it orders the
points by reachability and adapts the neighbourhood threshold, so clusters of different densities can be found in
one pass. Clusters are extracted with the `xi` method (`xi` = the minimum relative drop in reachability that marks
a cluster boundary). To keep it tractable the reachability search is capped at 520 km, the largest `eps` of the
DBSCAN grid.

**From the HDBSCAN talk (Healy & McInnes).** HDBSCAN is DBSCAN run over every `eps` at once: its single-linkage tree
of mutual reachability distances contains every DBSCAN clustering as a flat cut, and the condensed tree keeps only
the splits that leave at least `min_cluster_size` points on both sides. Clusters are chosen by *persistence*
(stability: how long and how large a cluster lives in the tree). The core distance (distance to the
`min_samples`-th neighbour) is a fast local density estimate, and GLOSH scores how much of an outlier each point is
relative to its cluster. All of these are stored below.

## 5. DBSCAN grid

In [4]:
# 24 eps x 11 min_samples per dataset. Labels are kept for every run; core flags are kept for the DBSCAN* check.
runs, labels, dbscan_core = [], {}, {}
t0 = time.time()
for name in DATASETS:
    for eps in ca.EPS_KM:
        for ms in ca.MIN_SAMPLES:
            lab, core = ca.fit_dbscan(X[name], eps, ms)
            rid = ca.run_id(name, "DBSCAN", eps_km=eps, min_samples=ms)
            labels[rid], dbscan_core[rid] = lab, core
            runs.append({"run_id": rid, "dataset": name, "algorithm": "DBSCAN", "eps_km": eps,
                         "min_cluster_size": np.nan, "min_samples": ms, "xi": np.nan, **ca.basic_metrics(lab)})
print(f"{sum(r['algorithm'] == 'DBSCAN' for r in runs)} DBSCAN runs in {time.time() - t0:.0f} s")

528 DBSCAN runs in 50 s


## 6. HDBSCAN grid

The single-linkage tree depends only on `min_samples`, so one tree per `min_samples` serves all 28
`min_cluster_size` values (labels identical to a full fit). The "default" column (`min_samples = min_cluster_size`)
needs one tree per `min_cluster_size`. For each run we keep labels, membership probabilities, GLOSH outlier scores,
the persistence of each cluster, and the cluster-level skeleton of the condensed tree.

In [5]:
hdbscan_extras, trees = {}, {}
t0 = time.time()
for name in DATASETS:
    for ms in [None] + ca.MIN_SAMPLES:
        if ms is not None:
            trees[(name, ms)] = ca.hdbscan_tree(X[name], ms, N_JOBS)
        for mcs in ca.MIN_CLUSTER_SIZE:
            tree = trees[(name, ms)] if ms is not None else ca.hdbscan_tree(X[name], mcs, N_JOBS)
            lab, prob, persistence, condensed, glosh = ca.hdbscan_from_tree(X[name], tree, mcs)
            rid = ca.run_id(name, "HDBSCAN", min_cluster_size=mcs, min_samples=ms)
            labels[rid] = lab
            skeleton = ca.condensed_clusters(condensed, lab)
            hdbscan_extras[rid] = {"probabilities": prob, "glosh": glosh, "persistence": persistence,
                                   "condensed": skeleton}
            runs.append({"run_id": rid, "dataset": name, "algorithm": "HDBSCAN", "eps_km": np.nan,
                         "min_cluster_size": mcs, "min_samples": np.nan if ms is None else ms, "xi": np.nan,
                         **ca.basic_metrics(lab),
                         "mean_persistence": float(np.mean(persistence)) if len(persistence) else np.nan,
                         "mean_glosh": float(np.mean(glosh)), "glosh_p95": float(np.percentile(glosh, 95))})
print(f"{sum(r['algorithm'] == 'HDBSCAN' for r in runs)} HDBSCAN runs in {time.time() - t0:.0f} s")

672 HDBSCAN runs in 86 s


### Check: `hdbscan` library vs scikit-learn

The first notebook used scikit-learn's HDBSCAN. The two implementations should agree up to how they break ties;
this dataset has many ties because hundreds of breweries share exact coordinates.

In [6]:
checks = []
for name in DATASETS:
    for mcs, ms in [(15, 4), (15, 15), (30, 10), (50, 25), (100, 8)]:
        sk = SklearnHDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="haversine").fit_predict(X[name])
        lib = labels[ca.run_id(name, "HDBSCAN", min_cluster_size=mcs, min_samples=ms)]
        checks.append({"dataset": name, "min_cluster_size": mcs, "min_samples": ms,
                       "clusters_sklearn": int(sk.max() + 1), "clusters_hdbscan_lib": int(lib.max() + 1),
                       "ari": adjusted_rand_score(sk, lib)})
hdbscan_check = pd.DataFrame(checks)
hdbscan_check.round(3)

/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,dataset,min_cluster_size,min_samples,clusters_sklearn,clusters_hdbscan_lib,ari
0,all_records,15,4,191,190,0.966
1,all_records,15,15,135,135,0.992
2,all_records,30,10,83,82,0.985
3,all_records,50,25,42,42,0.998
4,all_records,100,8,20,20,0.999
5,unique_records,15,4,186,185,0.980
6,unique_records,15,15,134,135,0.971
7,unique_records,30,10,83,82,0.983
8,unique_records,50,25,42,42,0.998
9,unique_records,100,8,20,20,1.000


## 7. DBSCAN as a flat cut of the HDBSCAN tree

For every `eps` and `min_samples`, cut the HDBSCAN tree built with the same `min_samples` at `eps` and compare with
the DBSCAN run. The talk's claim is that the cut is DBSCAN* — DBSCAN without border points — so the two should
agree exactly on DBSCAN's core points and differ only on border points.

In [7]:
star_rows = []
for name in DATASETS:
    for eps in ca.EPS_KM:
        for ms in ca.MIN_SAMPLES:
            rid = ca.run_id(name, "DBSCAN", eps_km=eps, min_samples=ms)
            db, core = labels[rid], dbscan_core[rid]
            cut = ca.dbscan_star_cut(trees[(name, ms)], eps)
            star_rows.append({"dataset": name, "eps_km": eps, "min_samples": ms,
                              "ari_all_points": adjusted_rand_score(db, cut),
                              "ari_core_points": adjusted_rand_score(db[core], cut[core]) if core.sum() > 1 else np.nan,
                              "border_points": int(((db != -1) & ~core).sum()),
                              "clusters_dbscan": int(db.max() + 1), "clusters_tree_cut": int(cut.max() + 1)})
dbscan_star = pd.DataFrame(star_rows)
print(dbscan_star[["ari_all_points", "ari_core_points"]].describe().round(4))

       ari_all_points  ari_core_points
count        528.0000            528.0
mean           0.9201              1.0
std            0.1031              0.0
min            0.6655              1.0
25%            0.8598              1.0
50%            0.9781              1.0
75%            0.9974              1.0
max            1.0000              1.0


## 8. OPTICS

One OPTICS fit per `min_samples` gives the reachability ordering; the three `xi` values are extracted from it. The
reachability plot and the core distances (local density estimates for contour maps) are stored per fit.

In [8]:
optics_store = []
t0 = time.time()
for name in DATASETS:
    for ms in ca.MIN_SAMPLES:
        model = ca.fit_optics(X[name], ms)
        optics_store.append({"dataset": name, "min_samples": ms,
                             "ordering": ca.encode(model.ordering_, np.int16),
                             "reachability_km": ca.encode(model.reachability_ * ca.R, np.float32),
                             "core_distance_km": ca.encode(model.core_distances_ * ca.R, np.float32)})
        for xi in ca.OPTICS_XI:
            lab = ca.optics_labels(model, ms, xi)
            rid = ca.run_id(name, "OPTICS", min_samples=ms, xi=xi)
            labels[rid] = lab
            runs.append({"run_id": rid, "dataset": name, "algorithm": "OPTICS", "eps_km": np.nan,
                         "min_cluster_size": np.nan, "min_samples": ms, "xi": xi, **ca.basic_metrics(lab)})
print(f"{sum(r['algorithm'] == 'OPTICS' for r in runs)} OPTICS runs from {len(optics_store)} fits "
      f"in {time.time() - t0:.0f} s")

/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


... (5 outputs left out)

/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]
/home/rahmat/datah/geolysis/.venv/lib/python3.13/site-packages/sklearn/cluster/_optics.py:1086: RuntimeWarning: divide by zero encountered in divide
  ratio = reachability_plot[:-1] / reachability_plot[1:]


66 OPTICS runs from 22 fits in 117 s


## 9. Geography of every run

For each run, every cluster is described (center, spread, hull area and density, states and countries spanned,
dominant place and brewery type, hubs inside) and the run is summarised: typical compactness, how many clusters
cross a state or country line, and how many records sit in clusters that merge distinct hubs.

In [9]:
def summarise(df, chunk, state_caps):
    # run summaries for a chunk of (run_id, labels) of one dataset; the dataset travels to the worker once
    out = {}
    for rid, lab in chunk:
        t = ca.cluster_table(df, lab, HUB_MIN_RECORDS, HUB_MIN_INSIDE, HUB_MIN_SEP_KM, state_capitals=state_caps)
        out[rid] = ca.run_geo_summary(t, len(df))
    return out


t0 = time.time()
jobs = []
for name in DATASETS:
    items = [(rid, lab) for rid, lab in labels.items() if rid.startswith(name + "|")]
    for i in range(N_JOBS):
        jobs.append(delayed(summarise)(frames[name], items[i::N_JOBS], ref["state"]))
geo = {k: v for part in Parallel(n_jobs=N_JOBS)(jobs) for k, v in part.items()}
print(f"geography of {len(geo)} runs in {time.time() - t0:.0f} s")

geography of 1266 runs in 34 s


## 10. Stability across neighbouring parameters

A grouping that only appears under one unusual setting deserves suspicion. For every run we compare it with the
runs one grid step away along each parameter (e.g. DBSCAN 45 km / 15 against 40 and 50 km at 15, and 45 km at 12 and
20): the mean adjusted Rand index (ARI) of the whole labelling, and for every cluster the best Jaccard overlap with a
cluster of each neighbour (1 = the same breweries reappear). A cluster that recurs with Jaccard ≥ 0.5 on average
counts as stable.

In [10]:
run_table = pd.DataFrame(runs)
stab = []
for name in DATASETS:
    for alg, axes in [("DBSCAN", [("eps_km", ca.EPS_KM), ("min_samples", ca.MIN_SAMPLES)]),
                      ("HDBSCAN", [("min_cluster_size", ca.MIN_CLUSTER_SIZE), ("min_samples", ca.MIN_SAMPLES)]),
                      ("OPTICS", [("min_samples", ca.MIN_SAMPLES), ("xi", ca.OPTICS_XI)])]:
        sub = run_table[(run_table["dataset"] == name) & (run_table["algorithm"] == alg)].copy()
        if alg == "HDBSCAN":  # the default column (min_samples = min_cluster_size) only steps along min_cluster_size
            num, dflt = sub[sub["min_samples"].notna()], sub[sub["min_samples"].isna()].assign(min_samples=-1)
            nb = {**ca.neighbours(num, axes), **ca.neighbours(dflt, [axes[0], ("min_samples", [-1])])}
        else:
            nb = ca.neighbours(sub, axes)
        stab.append(ca.stability(labels, nb))
stability = pd.concat(stab, ignore_index=True)
run_table = run_table.merge(stability, on="run_id").merge(
    pd.DataFrame.from_dict(geo, orient="index").rename_axis("run_id").reset_index(), on="run_id")
run_table["diagnostic"] = run_table.apply(lambda r: "; ".join(
    w for w, c in [("fewer_than_two_clusters", r["cluster_count"] < 2), ("very_high_noise", r["noise_share_pct"] >= 80),
                   ("dominant_cluster", r["largest_cluster_share_pct"] >= 80), ("many_clusters", r["cluster_count"] >= 100),
                   ("merges_distinct_hubs", r["clusters_merging_hubs"] > 0)] if c) or "review_candidate", axis=1)
run_table[["run_id", "cluster_count", "noise_share_pct", "mean_neighbour_ari", "median_cluster_recurrence",
           "clusters_merging_hubs", "records_in_hub_merging_clusters_%"]].head(10).round(3)

,run_id,cluster_count,noise_share_pct,mean_neighbour_ari,median_cluster_recurrence,clusters_merging_hubs,records_in_hub_merging_clusters_%
0,all_records|DBSCAN|eps=25|ms=4,289,15.820,0.776,0.931,9,40.335
1,all_records|DBSCAN|eps=25|ms=5,221,19.477,0.818,0.971,9,38.427
2,all_records|DBSCAN|eps=25|ms=6,195,22.070,0.816,0.962,8,36.109
3,all_records|DBSCAN|eps=25|ms=8,160,26.465,0.839,0.952,6,26.307
4,all_records|DBSCAN|eps=25|ms=10,133,30.523,0.854,0.962,5,24.030
5,all_records|DBSCAN|eps=25|ms=12,112,34.696,0.830,0.955,5,22.302
6,all_records|DBSCAN|eps=25|ms=15,98,40.251,0.790,0.874,3,9.928
7,all_records|DBSCAN|eps=25|ms=20,72,50.042,0.770,0.918,1,2.171
8,all_records|DBSCAN|eps=25|ms=25,53,57.388,0.812,0.933,1,2.129
9,all_records|DBSCAN|eps=25|ms=30,42,63.459,0.857,0.956,0,0.000


## 11. Save

New tables only (`allruns_*`); the first notebook's tables are not touched. Label arrays are stored zlib-compressed
in the record order of `allruns_records` (int16; -1 = noise).

In [11]:
records = pd.concat([pd.DataFrame({"dataset": n, "position": np.arange(len(f)), "id": f["id"].to_numpy(),
                                   "coordinate_status": f["coordinate_status"].to_numpy()})
                     for n, f in frames.items()], ignore_index=True)
label_rows = pd.DataFrame({"run_id": list(labels), "labels": [ca.encode(v, np.int16) for v in labels.values()]})
extras = pd.DataFrame([{"run_id": rid, "probabilities": ca.encode(np.round(e["probabilities"] * 255), np.uint8),
                        "glosh": ca.encode(np.round(e["glosh"] * 255), np.uint8), "persistence": ca.encode(e["persistence"], np.float32),
                        "condensed_columns": ",".join(e["condensed"].columns),
                        "condensed": ca.encode(e["condensed"].to_numpy(dtype=np.float64), np.float32)}
                       for rid, e in hdbscan_extras.items()])
optics_tbl = pd.DataFrame(optics_store)
settings = pd.DataFrame([
    {"key": "hub_min_records", "value": HUB_MIN_RECORDS}, {"key": "hub_min_inside", "value": HUB_MIN_INSIDE},
    {"key": "hub_min_sep_km", "value": HUB_MIN_SEP_KM}, {"key": "optics_max_eps_km", "value": ca.OPTICS_MAX_EPS_KM},
    {"key": "earth_radius_km", "value": ca.R}, {"key": "hdbscan_library", "value": "hdbscan " + version("hdbscan")},
    {"key": "scikit_learn", "value": version("scikit-learn")},
    {"key": "min_samples_convention", "value": "scikit-learn (point counts itself); hdbscan receives min_samples - 1"},
    {"key": "corrections", "value": json.dumps(corrections["action"].value_counts().to_dict())}])
tables = {"allruns_records": records, "allruns_runs": run_table, "allruns_labels": label_rows,
          "allruns_hdbscan_extras": extras, "allruns_optics": optics_tbl, "allruns_dbscan_star": dbscan_star,
          "allruns_hdbscan_check": hdbscan_check, "allruns_settings": settings.astype(str)}
with sqlite3.connect(RESULTS_DB) as con:
    for tname, t in tables.items():
        t.to_sql(tname, con, if_exists="replace", index=False)
    verification = pd.DataFrame([{"table_name": tname, "expected_rows": len(t),
                                  "saved_rows": con.execute(f'SELECT COUNT(*) FROM "{tname}"').fetchone()[0]}
                                 for tname, t in tables.items()])
    verification["verified"] = verification["expected_rows"] == verification["saved_rows"]
    verification.to_sql("allruns_verification", con, if_exists="replace", index=False)
    con.execute("VACUUM")
print(verification.to_string(index=False))
print(f"clustering_results.db: {RESULTS_DB.stat().st_size / 1e6:.1f} MB")

            table_name  expected_rows  saved_rows  verified
       allruns_records          18975       18975      True
          allruns_runs           1266        1266      True
        allruns_labels           1266        1266      True
allruns_hdbscan_extras            672         672      True
        allruns_optics             22          22      True
   allruns_dbscan_star            528         528      True
 allruns_hdbscan_check             10          10      True
      allruns_settings              9           9      True
clustering_results.db: 31.5 MB


## 12. What the runs show

In [12]:
# Headline numbers for the write-up below.
rt = run_table[run_table["dataset"] == "all_records"]
summary = rt.groupby("algorithm").agg(runs=("run_id", "size"), median_clusters=("cluster_count", "median"),
                                      median_noise_pct=("noise_share_pct", "median"),
                                      runs_merging_hubs=("clusters_merging_hubs", lambda s: int((s > 0).sum())),
                                      median_neighbour_ari=("mean_neighbour_ari", "median"),
                                      median_cluster_recurrence=("median_cluster_recurrence", "median"))
print(summary.round(3).to_string())
db = rt[rt["algorithm"] == "DBSCAN"]
first_merge = db[db["clusters_merging_hubs"] > 0].groupby("min_samples")["eps_km"].min()
print("\nSmallest DBSCAN eps (km) at which a cluster merges distinct hubs, per min_samples:")
print(first_merge.to_string())
print("\nDBSCAN* vs DBSCAN on core points: min ARI", round(dbscan_star["ari_core_points"].min(), 4))

           runs  median_clusters  median_noise_pct  runs_merging_hubs  median_neighbour_ari  median_cluster_recurrence
algorithm                                                                                                             
DBSCAN      264             29.0             2.298                262                 0.937                      0.975
HDBSCAN     336             17.5            23.324                322                 0.875                      0.983
OPTICS       33            217.0            41.210                 12                 0.651                      0.857

Smallest DBSCAN eps (km) at which a cluster merges distinct hubs, per min_samples:
min_samples
4.0     25.0
5.0     25.0
6.0     25.0
8.0     25.0
10.0    25.0
12.0    25.0
15.0    25.0
20.0    25.0
25.0    25.0
30.0    30.0
32.0    30.0

DBSCAN* vs DBSCAN on core points: min ARI 1.0


### Reading the results (all_records; unique_records gives the same picture)

**The inputs were cleaned first.** 28 records were caught by the coordinate check: 8 had latitude and longitude
swapped and are swapped back, one (Mescan Brewery in Mayo, Ireland) had a longitude with the wrong sign that put it in
northern Germany and is reversed, one Liefmans record takes the position of its identical twin, and 18 records more
than 50 km from the city they name are kept but flagged. Every run below uses these corrected positions.

**HDBSCAN really is DBSCAN at every radius.** Cutting the exact HDBSCAN tree at a given `eps` reproduces the DBSCAN
run on its core points in all 528 settings, with one exception: a DBSCAN cluster built around a single core point.
The tree cut needs at least two points to form a cluster, so it leaves that core point as noise. Such clusters
account for the whole difference in cluster count (up to 15 per setting, checked in all 528), and the ARI on core
points never falls below 0.99997. Otherwise the two differ only on border points (median ARI over all points 0.978),
which DBSCAN attaches to a cluster and the tree cut leaves as noise. One tree per
`min_samples` therefore holds the whole DBSCAN grid, which is the point the HDBSCAN talk makes. This only holds with
the exact minimum spanning tree; the library's default approximate tree broke it in 156 of the 528 settings.

**The two HDBSCAN implementations agree.** The `hdbscan` library and scikit-learn give ARI 0.96–1.00 on the checked
settings; what differs is how ties are broken among breweries stacked on identical coordinates.

**A single radius chains brewery hubs together.** With `min_samples = 15`, DBSCAN already joins Brooklyn to hubs more
than 300 km away along the Northeast corridor at 30 km. At 45 km one cluster runs from Brussels to München (25 states,
hubs 690 km apart) and another from Baltimore to Boston (772 km). Only 2 of 264 DBSCAN settings keep every pair of
hubs more than 100 km apart in separate clusters (25 km with `min_samples` 30 or 32), and they label 63–66% of all
breweries as noise. At 25 km the merges are still metropolitan (San Diego with Los Angeles, Seattle with Tacoma and
Bellingham, the Bay Area: hubs at most 175 km apart); from about 30–45 km they become corridors and whole regions.
For this project that matters more than the size of the largest cluster: a cluster that spans Belgium, the
Netherlands and Bavaria is not a brewery hub, it is a chain of them.

**HDBSCAN avoids most chains, at the price of noise.** 14 of 336 HDBSCAN settings merge no hubs at all (15 on
unique_records). All of them have `min_cluster_size` ≤ 15 and `min_samples` ≤ 12 (or the default), and they leave
25–38% of breweries as noise. The first notebook's HDBSCAN_A (`min_cluster_size` 15, `min_samples` 4) is one of them:
190 clusters, 24.8% noise, a mean ARI of 0.70 with its neighbouring settings and a median cluster recurrence of 0.89.

**OPTICS finds the smallest and tightest groups.** A median of 217 clusters with a typical within-cluster distance of
about 11 km; 21 of its 33 runs merge no hubs, but it leaves 31–58% of breweries as noise in those runs and it is the
least stable across neighbouring settings (median ARI 0.65).

**Stability has to be read with geography.** DBSCAN looks the most stable (median neighbour ARI 0.94) precisely
because its large merged clusters barely change when `eps` moves one step. Stability alone rewards over-merging; it
is informative only together with hub merges, noise and compactness.

**No setting wins on every measure.** Fewer merged hubs costs noise; less noise costs merged hubs. The dashboard
ranks every run on these measures and shows any of them on the map, so the choice can be argued run by run.
